# -----------------------------------------------------
# Libraries
# -----------------------------------------------------

In [15]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

# -----------------------------------------------------
# PATH CONFIGURATION
# -----------------------------------------------------

In [16]:
# -----------------------------------------------------
# PATH CONFIGURATION
# -----------------------------------------------------

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))

from config.paths import RAW_DIR, PROCESSED_DIR

YEARS = [2018, 2024]

# -----------------------------------------------------
# TAX BRACKETS BY YEAR
# -----------------------------------------------------

In [8]:
# -----------------------------------------------------
# TAX BRACKETS BY YEAR
# -----------------------------------------------------

TAX_BRACKETS = {
    2018: [
        {"li": 0.01, "ls": 8952.49, "cf": 0, "tm": 0.0192},
        {"li": 8952.50, "ls": 75984.55, "cf": 171.88, "tm": 0.064},
        {"li": 75984.56, "ls": 133536.07, "cf": 4461.94, "tm": 0.1088},
        {"li": 133536.08, "ls": 155229.80, "cf": 10723.55, "tm": 0.16},
        {"li": 155229.81, "ls": 185852.57, "cf": 14194.54, "tm": 0.1792},
        {"li": 185852.58, "ls": 374837.88, "cf": 19682.13, "tm": 0.2136},
        {"li": 374837.89, "ls": 590795.99, "cf": 60049.40, "tm": 0.2352},
        {"li": 590796.00, "ls": 1127926.84, "cf": 110842.74, "tm": 0.30},
        {"li": 1127926.85, "ls": 1503902.46, "cf": 271981.99, "tm": 0.32},
        {"li": 1503902.47, "ls": 4511707.37, "cf": 392294.17, "tm": 0.34},
        {"li": 4511707.38, "ls": np.inf, "cf": 1414947.85, "tm": 0.35},
    ],

    2024: [
        {"li": 0.01, "ls": 8952.49, "cf": 0, "tm": 0.0192},
        {"li": 8952.50, "ls": 75984.55, "cf": 171.88, "tm": 0.064},
        {"li": 75984.56, "ls": 133536.07, "cf": 4461.94, "tm": 0.1088},
        {"li": 133536.08, "ls": 155229.80, "cf": 10723.55, "tm": 0.16},
        {"li": 155229.81, "ls": 185852.57, "cf": 14194.54, "tm": 0.1792},
        {"li": 185852.58, "ls": 374837.88, "cf": 19682.13, "tm": 0.2136},
        {"li": 374837.89, "ls": 590795.99, "cf": 60049.40, "tm": 0.2352},
        {"li": 590796.00, "ls": 1127926.84, "cf": 110842.74, "tm": 0.30},
        {"li": 1127926.85, "ls": 1503902.46, "cf": 271981.99, "tm": 0.32},
        {"li": 1503902.47, "ls": 4511707.37, "cf": 392294.17, "tm": 0.34},
        {"li": 4511707.38, "ls": np.inf, "cf": 1414947.85, "tm": 0.35},
    ]
}

# -----------------------------------------------------
# FUNCTION: GROSS-UP CALCULATION
# -----------------------------------------------------

In [9]:
# -----------------------------------------------------
# FUNCTION: GROSS-UP CALCULATION
# -----------------------------------------------------

def calculate_market_income(df, brackets):

    df["market_income"] = np.nan

    for b in brackets:

        gross_temp = (
            df["net_market_income"]
            + b["cf"]
            - b["tm"] * b["li"]
        ) / (1 - b["tm"])

        mask = (
            df["market_income"].isna()
        ) & (
            gross_temp >= b["li"]
        ) & (
            gross_temp <= b["ls"]
        )

        df.loc[mask, "market_income"] = gross_temp

    return df

# -----------------------------------------------------
# MAIN PIPELINE
# -----------------------------------------------------

In [17]:

# -----------------------------------------------------
# MAIN PIPELINE
# -----------------------------------------------------

all_years = []

for year in YEARS:

    print("\n========================")
    print(f"Processing {year}")
    print("========================")

    # -----------------------------
    # LOAD POBLACION
    # -----------------------------

    file_pob = RAW_DIR / f"poblacion_{year}.csv"

    df_pob = pd.read_csv(file_pob)

    print("Population rows:", len(df_pob))

    # -----------------------------
    # LOAD INGRESOS
    # -----------------------------

    file_ing = RAW_DIR / f"ingresos_{year}.csv"

    df_ing = pd.read_csv(file_ing)

    df_ing = df_ing[
        df_ing["clave"] == "P001"
    ].copy()

    df_ing = df_ing.rename(
        columns={
            "ing_tri": "net_market_income"
        }
    )

    # -----------------------------
    # CALCULATE MARKET INCOME
    # -----------------------------

    brackets = TAX_BRACKETS[year]

    df_ing = calculate_market_income(
        df_ing,
        brackets
    )

    # -----------------------------
    # KEEP ONLY RELEVANT COLUMNS
    # -----------------------------

    df_income = df_ing[
        [
            "folioviv",
            "foliohog",
            "numren",
            "market_income",
            "net_market_income"
        ]
    ]

    # -----------------------------
    # MERGE WITH POBLACION
    # -----------------------------

    df_year = df_pob.merge(
        df_income,
        on=[
            "folioviv",
            "foliohog",
            "numren"
        ],
        how="left"
    )

    # -----------------------------
    # ADD YEAR
    # -----------------------------

    df_year["year"] = year

    # -----------------------------
    # PREVIEW
    # -----------------------------

    print("\nPreview")

    display(
        df_year[
            [
                "folioviv",
                "foliohog",
                "numren",
                "year",
                "market_income",
                "net_market_income"
            ]
        ].head()
    )

    all_years.append(df_year)

# -----------------------------------------------------
# STACK YEARS
# -----------------------------------------------------

df_final = pd.concat(
    all_years,
    axis=0,
    ignore_index=True
)

print("\nFinal stacked dataset size:")

print(len(df_final))

print("\nYears distribution:")

print(
    df_final["year"].value_counts()
)

# -----------------------------------------------------
# EXPORT
# -----------------------------------------------------

output_file = (
    PROCESSED_DIR
    / "net market and market income.csv"
)

df_final.to_csv(
    output_file,
    index=False
)

print("\nSaved file:")

print(output_file.name)


Processing 2018


/var/folders/lw/wsksmjcd29scklkcylgwjjlr0000gn/T/ipykernel_3155/3695367124.py:19: DtypeWarning: Columns (0: disc1, 1: segpop, 2: atemed, 3: peso) have mixed types. Specify dtype option on import or set low_memory=False.
  df_pob = pd.read_csv(file_pob)


Population rows: 269206

Preview


,folioviv,foliohog,numren,year,market_income,net_market_income
0,100013601,1,1,2018,31097.339744,29508.19
1,100013601,1,2,2018,NaN,NaN
2,100013601,1,3,2018,24792.168803,23606.55
3,100013602,1,1,2018,NaN,NaN
4,100013602,1,2,2018,NaN,NaN



Processing 2024


/var/folders/lw/wsksmjcd29scklkcylgwjjlr0000gn/T/ipykernel_3155/3695367124.py:19: DtypeWarning: Columns (0: conyuge_id) have mixed types. Specify dtype option on import or set low_memory=False.
  df_pob = pd.read_csv(file_pob)


Population rows: 308598

Preview


/var/folders/lw/wsksmjcd29scklkcylgwjjlr0000gn/T/ipykernel_3155/3695367124.py:84: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_year["year"] = year


,folioviv,foliohog,numren,year,market_income,net_market_income
0,100001901,1,1,2024,51870.822650,48952.17
1,100001901,1,2,2024,30926.004274,29347.82
2,100001901,1,3,2024,NaN,NaN
3,100001901,1,4,2024,NaN,NaN
4,100001902,1,1,2024,49738.717949,46956.52



Final stacked dataset size:
577804

Years distribution:
year
2024    308598
2018    269206
Name: count, dtype: int64

Saved file:
net market and market income.csv
